## 22. عرضه و روند قیمت

تحلیل زمانی باید به تفکیک مناسب انجام شود:

- ماه
- شهر
- محله، در صورت کفایت نمونه
- نوع ملک
- رژیم قیمت

شاخص‌های پیشنهادی:

- تعداد آگهی خام
- تعداد آگهی پس از Deduplication
- میانه قیمت هر مترمربع
- تغییر ماه‌به‌ماه
- IQR قیمت
- تعداد مناطق دارای داده کافی

### محدودیت

تغییر تعداد آگهی می‌تواند ناشی از تغییر رفتار کاربران، پوشش پلتفرم، Duplicate یا فصل باشد.
آن را مستقیماً برابر با تغییر موجودی واقعی بازار در نظر نگیرید.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_feather("../Outputs/21_df.feather")

In [ ]:
import pandas as pd
import numpy as np

# ۱. بارگذاری داده
df = pd.read_feather("../Outputs/21_df.feather")

# -----------------------------
# ۱. آماده سازی و تعریف Duplicate
# -----------------------------

# تبدیل ستون قیمت به عدد
df['sale_price_per_sqm'] = pd.to_numeric(
    df['sale_price_per_sqm'],
    errors='coerce'
)

# ایجاد تاریخ ماهانه
df['month'] = df['created_at_month'].dt.to_period('M').dt.to_timestamp()

# پر کردن مقادیر Null احتمالی در ستون‌های duplicate با False
is_exact = df['is_exact_duplicate'].fillna(False)
is_probable = df['is_probable_duplicate_to_remove'].fillna(False)

# تعریف ستون duplicate اصلی (ترکیب شرط exact یا probable)
df['is_duplicate'] = is_exact | is_probable


# -----------------------------
# ۲. ستون های گروه بندی
# -----------------------------

group_cols = [
    'month',
    'city_slug',
    'neighborhood_slug',
    'cat2_slug'
]


# -----------------------------
# ۳. محاسبه تعداد آگهی (خام و Deduplicated)
# -----------------------------

# تعداد آگهی خام
ads_raw = df.groupby(
    group_cols,
    observed=True
).size().reset_index(name='ads_raw_count')

# تعداد آگهی پس از حذف Duplicateها
ads_dedup = df[~df['is_duplicate']].groupby(
    group_cols,
    observed=True
).size().reset_index(name='ads_dedup_count')

# ترکیب تعداد آگهی خام و یکتا
ads = ads_raw.merge(ads_dedup, on=group_cols, how='left')
ads['ads_dedup_count'] = ads['ads_dedup_count'].fillna(0).astype(int)


# -----------------------------
# ۴. قیمت معتبر (فقط از آگهی‌های non-duplicate)
# -----------------------------

price_df = df[
    (~df['is_duplicate']) &
    df['sale_price_per_sqm'].notna() &
    (df['sale_price_per_sqm'] > 0)
].copy()


# -----------------------------
# ۵. آمار قیمت
# -----------------------------

price_stats = price_df.groupby(
    group_cols,
    observed=True
)['sale_price_per_sqm'].agg(
    median_price_per_sqm='median',
    q1_price_per_sqm=lambda x: x.quantile(0.25),
    q3_price_per_sqm=lambda x: x.quantile(0.75),
    price_count='count'
).reset_index()


# -----------------------------
# ۶. IQR
# -----------------------------

price_stats['price_iqr'] = (
    price_stats['q3_price_per_sqm']
    -
    price_stats['q1_price_per_sqm']
)


# -----------------------------
# ۷. ترکیب آمار آگهی و قیمت
# -----------------------------

supply_trend = ads.merge(
    price_stats,
    on=group_cols,
    how='left'
)


# -----------------------------
# ۸. مرتب سازی
# -----------------------------

supply_trend = supply_trend.sort_values(
    [
        'city_slug',
        'neighborhood_slug',
        'cat2_slug',
        'month'
    ]
)


# -----------------------------
# ۹. تغییر ماهانه قیمت
# -----------------------------

trend_cols = [
    'city_slug',
    'neighborhood_slug',
    'cat2_slug'
]

supply_trend['price_mom_pct'] = (
    supply_trend
    .groupby(
        trend_cols,
        observed=True
    )['median_price_per_sqm']
    .pct_change()
    * 100
)


# -----------------------------
# ۱۰. تغییر ماهانه تعداد آگهی (بر اساس داده‌های Deduplicated)
# -----------------------------

supply_trend['ads_dedup_mom_pct'] = (
    supply_trend
    .groupby(
        trend_cols,
        observed=True
    )['ads_dedup_count']
    .pct_change()
    * 100
)


# -----------------------------
# ۱۱. کفایت نمونه
# -----------------------------

MIN_SAMPLE = 20

supply_trend['enough_data'] = (
    supply_trend['price_count'] >= MIN_SAMPLE
)

print(supply_trend.head())


In [ ]:
# # -----------------------------
# # 1. آماده سازی
# # -----------------------------

# df['sale_price_per_sqm'] = pd.to_numeric(
#     df['sale_price_per_sqm'],
#     errors='coerce'
# )

# df['month'] = df['created_at_month'].dt.to_period('M').dt.to_timestamp()


# is_exact = df['is_exact_duplicate'].fillna(False)
# is_probable = df['is_probable_duplicate_to_remove'].fillna(False)

# # تعریف ستون duplicate اصلی (ترکیب شرط exact یا probable)
# df['is_duplicate'] = is_exact | is_probable

# # -----------------------------
# # 2. ستون های گروه بندی
# # -----------------------------

# group_cols = [
#     'month',
#     'city_slug',
#     'neighborhood_slug',
#     'cat2_slug'
# ]


# # -----------------------------
# # 3. تعداد آگهی
# # -----------------------------

# ads = df.groupby(
#     group_cols,
#     observed=True
# ).size().reset_index(
#     name='ads_count'
# )


# # -----------------------------
# # 4. قیمت معتبر
# # -----------------------------

# price_df = df[
#     df['sale_price_per_sqm'].notna() &
#     (df['sale_price_per_sqm'] > 0)
# ].copy()


# # -----------------------------
# # 5. آمار قیمت
# # -----------------------------

# price_stats = price_df.groupby(
#     group_cols,
#     observed=True
# )['sale_price_per_sqm'].agg(
#     median_price_per_sqm='median',
#     q1_price_per_sqm=lambda x: x.quantile(0.25),
#     q3_price_per_sqm=lambda x: x.quantile(0.75),
#     price_count='count'
# ).reset_index()


# # -----------------------------
# # 6. IQR
# # -----------------------------

# price_stats['price_iqr'] = (
#     price_stats['q3_price_per_sqm']
#     -
#     price_stats['q1_price_per_sqm']
# )


# # -----------------------------
# # 7. ترکیب
# # -----------------------------

# supply_trend = ads.merge(
#     price_stats,
#     on=group_cols,
#     how='left'
# )


# # -----------------------------
# # 8. مرتب سازی
# # -----------------------------

# supply_trend = supply_trend.sort_values(
#     [
#         'city_slug',
#         'neighborhood_slug',
#         'cat2_slug',
#         'month'
#     ]
# )


# # -----------------------------
# # 9. تغییر ماهانه قیمت
# # -----------------------------

# trend_cols = [
#     'city_slug',
#     'neighborhood_slug',
#     'cat2_slug'
# ]

# supply_trend['price_mom_pct'] = (
#     supply_trend
#     .groupby(
#         trend_cols,
#         observed=True
#     )['median_price_per_sqm']
#     .pct_change()
#     * 100
# )


# # -----------------------------
# # 10. تغییر ماهانه تعداد آگهی
# # -----------------------------

# supply_trend['ads_mom_pct'] = (
#     supply_trend
#     .groupby(
#         trend_cols,
#         observed=True
#     )['ads_count']
#     .pct_change()
#     * 100
# )


# # -----------------------------
# # 11. کفایت نمونه
# # -----------------------------

# MIN_SAMPLE = 20

# supply_trend['enough_data'] = (
#     supply_trend['price_count'] >= MIN_SAMPLE
# )


# print(supply_trend.head())

In [ ]:
supply_trend.to_csv(
    '../Outputs/supply_trend.csv',
    index=False,
    encoding='utf-8-sig'
)